# CAS Exam 5: Expected Claims Method (Using Friedland Industry Auto Data)

This notebook uses:
- `C:/Users/cphel/Documents/code/exam_5/chainladder-python/chainladder/utils/data/friedland_us_industry_auto.csv`

Learning goals:
- Show expected claims method concepts with the same dataset used in the chain ladder notebook.
- Use `chainladder.Triangle` for the claims side, then apply ECR/pure-premium priors for ultimate and IBNR.
- Compare provided ECR vs derived ECR approaches with actuarial considerations.


## Formula-Sheet Core Equations

Premium basis:
- Ultimate Claims = (Expected Claims / Earned Premium) x Earned Premium
- ECR = Expected Claims / Earned Premium

Exposure basis:
- Ultimate Claims = (Expected Claims / Earned Exposure) x Earned Exposure
- Expected Pure Premium = Expected Claims / Earned Exposure

Interpretation:
- Expected claims method uses a priori assumptions (ECR/pure premium) more heavily than immature emergence patterns.


In [17]:
from __future__ import annotations

from pathlib import Path
import sys

import chainladder as cl
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import (
    adjust_selected_ecr,
    build_environment_impact_table,
    expected_claims_from_exposure,
    expected_claims_from_premium,
    ibnr_from_expected_claims,
    implied_ecr,
    selected_ecr_from_history,
    triangle_to_frame,
)

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto.csv'
raw = pd.read_csv(DATA_PATH)

triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Claims', 'Reported Claims'],
    cumulative=True,
)
paid_triangle = triangle['Paid Claims']
reported_triangle = triangle['Reported Claims']

raw.head(), {'triangle_shape': triangle.shape, 'valuation_date': str(triangle.valuation_date)}


(   Accident Year  Calendar Year  Paid Claims  Reported Claims
 0           1998           1998     18539254         37017487
 1           1998           1999     33231039         43169009
 2           1998           2000     40062008         45568919
 3           1998           2001     43892039         46784558
 4           1998           2002     45896535         47337318,
 {'triangle_shape': (1, 2, 10, 10),
  'valuation_date': '2007-12-31 23:59:59.999999999'})

In [18]:
paid_long = triangle_to_frame(paid_triangle, origin_as_datetime=False).reset_index()
reported_long = triangle_to_frame(reported_triangle, origin_as_datetime=False).reset_index()
paid_matrix = paid_long.pivot(index='origin', columns='development', values='Paid Claims').sort_index().sort_index(axis=1)
reported_matrix = reported_long.pivot(index='origin', columns='development', values='Reported Claims').sort_index().sort_index(axis=1)

latest_paid = paid_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_reported = reported_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_age = paid_matrix.notna().iloc[:, ::-1].idxmax(axis=1)

claim_inputs = pd.DataFrame(
    {
        'LatestPaid': latest_paid.values,
        'LatestReported': latest_reported.values,
        'LatestDevelopmentAge': latest_age.values,
    },
    index=latest_paid.index.year,
)
claim_inputs.index.name = 'AccidentYear'
claim_inputs


,LatestPaid,LatestReported,LatestDevelopmentAge
AccidentYear,,,
1998,47644187.0,47742304.0,120
1999,51000534.0,51185767.0,108
2000,54533225.0,54837929.0,96
2001,55878421.0,56299562.0,84
2002,57807215.0,58592712.0,72
2003,55930654.0,57565344.0,60
2004,53774672.0,56976657.0,48
2005,50644994.0,56786410.0,36
2006,43606497.0,54641339.0,24


## Build Priors (Earned Premium / Exposure) Aligned to This Dataset

This CSV has paid and reported claims but not premium/exposure. For expected-claims demonstration, we add explicit prior assumptions by AY:
- pricing ECR assumption to infer earned premium scale,
- average premium per exposure to infer earned exposure.

These priors are the central modeling choice in the expected claims method.


In [19]:
ay = claim_inputs.index

pricing_ecr_assumption = pd.Series(
    [0.74, 0.74, 0.745, 0.75, 0.755, 0.76, 0.765, 0.77, 0.775, 0.78],
    index=ay,
    dtype=float,
)
earned_premium = claim_inputs['LatestReported'] / pricing_ecr_assumption

avg_premium_per_exposure = pd.Series(
    [4700, 4750, 4800, 4850, 4900, 4950, 5000, 5050, 5100, 5150],
    index=ay,
    dtype=float,
)
earned_exposure = earned_premium / avg_premium_per_exposure

prior_inputs = pd.DataFrame(
    {
        'PricingECRAssumption': pricing_ecr_assumption,
        'EarnedPremium': earned_premium,
        'AvgPremiumPerExposure': avg_premium_per_exposure,
        'EarnedExposure': earned_exposure,
    }
)
prior_inputs


,PricingECRAssumption,EarnedPremium,AvgPremiumPerExposure,EarnedExposure
AccidentYear,,,,
1998,0.740,6.451663e+07,4700.0,13726.941921
1999,0.740,6.916996e+07,4750.0,14562.095875
2000,0.745,7.360796e+07,4800.0,15334.991331
2001,0.750,7.506608e+07,4850.0,15477.542818
2002,0.755,7.760624e+07,4900.0,15838.008380
2003,0.760,7.574387e+07,4950.0,15301.792663
2004,0.765,7.447929e+07,5000.0,14895.858039
2005,0.770,7.374858e+07,5050.0,14603.680082
2006,0.775,7.050495e+07,5100.0,13824.500696


## Provided ECR Case and Exposure-Basis Cross-Check

Exam guideline reminder:
- If provided ECR is not tied to a specific AY, treat it as broadly applicable unless stated otherwise.

We project expected ultimate with a provided ECR, then show the equivalent exposure-basis framing.


In [20]:
provided_ecr = 0.78
expected_ultimate_provided = expected_claims_from_premium(earned_premium, provided_ecr)

selected_pure_premium = float(expected_ultimate_provided.sum() / earned_exposure.sum())
expected_ultimate_exposure = expected_claims_from_exposure(earned_exposure, selected_pure_premium)

provided_view = pd.DataFrame(
    {
        'EarnedPremium': earned_premium,
        'ExpectedUltimate_ProvidedECR': expected_ultimate_provided,
        'ExpectedUltimate_ExposureBasis': expected_ultimate_exposure,
    }
)
provided_view.loc['Total'] = provided_view.sum()
provided_view


,EarnedPremium,ExpectedUltimate_ProvidedECR,ExpectedUltimate_ExposureBasis
AccidentYear,,,
1998,6.451663e+07,5.032297e+07,5.268578e+07
1999,6.916996e+07,5.395257e+07,5.589121e+07
2000,7.360796e+07,5.741421e+07,5.885768e+07
2001,7.506608e+07,5.855154e+07,5.940481e+07
2002,7.760624e+07,6.053287e+07,6.078833e+07
2003,7.574387e+07,5.908022e+07,5.873026e+07
2004,7.447929e+07,5.809385e+07,5.717223e+07
2005,7.374858e+07,5.752390e+07,5.605082e+07
2006,7.050495e+07,5.499386e+07,5.306022e+07


## Determine ECR from Historical Experience

Formula-sheet process:
1. Adjust historical data as needed.
2. Calculate claim ratios.
3. Select ECR (arithmetic, median, volume-weighted).

Here we use mature AYs (development age >= 84) and latest reported as a proxy for near-ultimate historical claims.


In [21]:
mature_mask = claim_inputs['LatestDevelopmentAge'] >= 84
historical_claims_proxy = claim_inputs.loc[mature_mask, 'LatestReported']
historical_premium = earned_premium.loc[mature_mask]

historical_ecr = implied_ecr(historical_claims_proxy, historical_premium)
selected_ecrs = pd.Series(
    {
        'ArithmeticECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='arithmetic'),
        'MedianECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='median'),
        'VolumeWeightedECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='volume_weighted'),
    }
)

historical_review = pd.DataFrame(
    {
        'HistoricalClaimsProxy': historical_claims_proxy,
        'HistoricalPremium': historical_premium,
        'ImpliedHistoricalECR': historical_ecr,
    }
)
historical_review, selected_ecrs


(              HistoricalClaimsProxy  HistoricalPremium  ImpliedHistoricalECR
 AccidentYear                                                                
 1998                     47742304.0       6.451663e+07                 0.740
 1999                     51185767.0       6.916996e+07                 0.740
 2000                     54837929.0       7.360796e+07                 0.745
 2001                     56299562.0       7.506608e+07                 0.750,
 ArithmeticECR        0.743750
 MedianECR            0.742500
 VolumeWeightedECR    0.743962
 dtype: float64)

## Apply ECR Level Adjustment and Project IBNR

General guideline from the sheet:
- If premium level differs between historical calibration and projection AY, adjust ECR level before applying.

Below uses volume-weighted historical ECR with a 3% level adjustment.


In [22]:
base_ecr = float(selected_ecrs['VolumeWeightedECR'])
level_adjustment = 1.03
adjusted_ecr = adjust_selected_ecr(base_ecr, level_adjustment=level_adjustment)

expected_ultimate_derived = expected_claims_from_premium(earned_premium, adjusted_ecr)
paid_to_date = claim_inputs['LatestPaid']
ibnr_derived = ibnr_from_expected_claims(expected_ultimate_derived, paid_to_date)

derived_projection = pd.DataFrame(
    {
        'PaidToDate': paid_to_date,
        'ExpectedUltimate_Derived': expected_ultimate_derived,
        'IBNR_Derived': ibnr_derived,
    }
)
derived_projection.loc['Total'] = derived_projection.sum()
pd.Series({'BaseECR': base_ecr, 'LevelAdjustment': level_adjustment, 'AdjustedECR': adjusted_ecr}), derived_projection


(BaseECR            0.743962
 LevelAdjustment    1.030000
 AdjustedECR        0.766281
 dtype: float64,
                PaidToDate  ExpectedUltimate_Derived  IBNR_Derived
 AccidentYear                                                     
 1998           47644187.0              4.943785e+07  1.793667e+06
 1999           51000534.0              5.300361e+07  2.003076e+06
 2000           54533225.0              5.640437e+07  1.871141e+06
 2001           55878421.0              5.752170e+07  1.643278e+06
 2002           57807215.0              5.946817e+07  1.660959e+06
 2003           55930654.0              5.804108e+07  2.110423e+06
 2004           53774672.0              5.707205e+07  3.297379e+06
 2005           50644994.0              5.651213e+07  5.867131e+06
 2006           43606497.0              5.402659e+07  1.042010e+07
 2007           27229969.0              4.799429e+07  2.076432e+07
 Total         498050368.0              5.494818e+08  5.143147e+07)

## Assumptions, Use Cases, and Environmental Impacts

Key assumptions:
1. A priori estimate is more reliable than immature emergence.
2. Current paid/reported to date may have limited predictive power for ultimate on young AYs.

Works well when:
- new line/territory has limited historical development credibility,
- operational changes reduce comparability of historical development,
- early maturity development factors are highly leveraged.


In [23]:
impact_table = build_environment_impact_table()
impact_table


,Description,Paid impact,Reported impact
0,Increase in exposure,No material effect if average accident date is...,No material effect if average accident date is...
1,Average accident date shifts forward,Underestimates ultimate (usually less than dev...,Underestimates ultimate (usually less than dev...
2,Increase claim ratios,"If not reflected in selected ECR, ultimates ar...","If not reflected in selected ECR, ultimates ar..."
3,Speedup in claim settlement rate,Overestimates ultimate (usually less than deve...,No material effect
4,Increase in case outstanding adequacy,No material effect,Overestimates ultimate (usually less than deve...
5,Change in product mix,Impacted when segments have different ECRs/dev...,Impacted when segments have different ECRs/dev...


## Provided vs Derived ECR Comparison

Educational takeaway:
- Provided ECR approach is straightforward and transparent but sensitive to whether the supplied ratio reflects current conditions.
- Derived ECR approach is anchored in historical calibration but sensitive to calibration window, mix changes, and level adjustments.


In [24]:
year_specific_ecr = pd.Series(
    [0.77, 0.77, 0.775, 0.78, 0.785, 0.79, 0.795, 0.80, 0.805, 0.81],
    index=ay,
    dtype=float,
)

expected_ultimate_year_specific = expected_claims_from_premium(earned_premium, year_specific_ecr)
ibnr_provided = ibnr_from_expected_claims(expected_ultimate_provided, paid_to_date)
ibnr_year_specific = ibnr_from_expected_claims(expected_ultimate_year_specific, paid_to_date)

comparison = pd.DataFrame(
    {
        'Ultimate_ProvidedECR': expected_ultimate_provided,
        'Ultimate_DerivedAdjustedECR': expected_ultimate_derived,
        'Ultimate_YearSpecificECR': expected_ultimate_year_specific,
        'IBNR_ProvidedECR': ibnr_provided,
        'IBNR_DerivedAdjustedECR': ibnr_derived,
        'IBNR_YearSpecificECR': ibnr_year_specific,
    }
)
comparison.loc['Total'] = comparison.sum()
comparison


,Ultimate_ProvidedECR,Ultimate_DerivedAdjustedECR,Ultimate_YearSpecificECR,IBNR_ProvidedECR,IBNR_DerivedAdjustedECR,IBNR_YearSpecificECR
AccidentYear,,,,,,
1998,5.032297e+07,4.943785e+07,4.967780e+07,2.678782e+06,1.793667e+06,2.033616e+06
1999,5.395257e+07,5.300361e+07,5.326087e+07,2.952031e+06,2.003076e+06,2.260332e+06
2000,5.741421e+07,5.640437e+07,5.704617e+07,2.880983e+06,1.871141e+06,2.512943e+06
2001,5.855154e+07,5.752170e+07,5.855154e+07,2.673123e+06,1.643278e+06,2.673123e+06
2002,6.053287e+07,5.946817e+07,6.092090e+07,2.725653e+06,1.660959e+06,3.113684e+06
2003,5.908022e+07,5.804108e+07,5.983766e+07,3.149567e+06,2.110423e+06,3.907006e+06
2004,5.809385e+07,5.707205e+07,5.921104e+07,4.319174e+06,3.297379e+06,5.436364e+06
2005,5.752390e+07,5.651213e+07,5.899887e+07,6.878902e+06,5.867131e+06,8.353874e+06
2006,5.499386e+07,5.402659e+07,5.675649e+07,1.138737e+07,1.042010e+07,1.314999e+07
